## PDF RAG


### Import libraries

In [2]:
! pip install langchain-experimental

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 KB 6.7 MB/s eta 0:00:00


In [3]:
# Imports
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

# Try to import MultiQueryRetriever - may need langchain-text-splitters
try:
    from langchain_community.retrievers.multi_query import MultiQueryRetriever
except ImportError:
    try:
        from langchain.retrievers.multi_query import MultiQueryRetriever
    except ImportError:
        MultiQueryRetriever = None
        print("Note: MultiQueryRetriever not available, will use basic retriever")

import os
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, Markdown

# Load environment variables from .env file
load_dotenv()

# Set environment variable for protobuf
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

Note: MultiQueryRetriever not available, will use basic retriever


### Load PDF

In [2]:
# Load PDF
local_path = "/workspace/pdfs/scammer-agent.pdf"
if local_path:
    loader = UnstructuredPDFLoader(file_path=local_path)
    data = loader.load()
    print(f"PDF loaded successfully: {local_path}")
else:
    print("Upload a PDF file")

PDF loaded successfully: /workspace/pdfs/scammer-agent.pdf


### Split text into chunks

In [3]:
# Split text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(data)
print(f"Text split into {len(chunks)} chunks")

Text split into 23 chunks


### Create vector database

In [4]:
# Create vector database
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2"),
    collection_name="local-rag"
)
print("Vector database created successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector database created successfully


### Set up LLM and Retrieval

In [33]:
# Initialize OpenAI LLM
# API key is loaded from .env file
API_KEY = os.getenv('OPENAI_API_KEY')
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.5,
    max_tokens=512
)

In [ ]:
# Query prompt template for MultiQueryRetriever
QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""You are an AI language model assistant. Your task is to generate 2
    different versions of the given user question to retrieve relevant documents from
    a vector database. By generating multiple perspectives on the user question, your
    goal is to help the user overcome some of the limitations of the distance-based
    similarity search. Provide these alternative questions separated by newlines.
    Original question: {question}""",
)

# Set up retriever - use MultiQueryRetriever if available, else use basic retriever
if MultiQueryRetriever is not None:
    try:
        retriever = MultiQueryRetriever.from_llm(
            vector_db.as_retriever(), 
            llm,
            prompt=QUERY_PROMPT
        )
        print("Using MultiQueryRetriever")
    except Exception as e:
        print(f"MultiQueryRetriever failed ({str(e)}), using basic retriever")
        retriever = vector_db.as_retriever(search_kwargs={"k": 5})
else:
    print("MultiQueryRetriever not available, using basic retriever")
    retriever = vector_db.as_retriever(search_kwargs={"k": 5})

### Create chain

In [24]:
# RAG prompt template
template = """Answer the question based ONLY on the following context:
{context}
Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

In [25]:
# Create chain
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

### Chat with PDF

In [26]:
def chat_with_pdf(question):
    """
    Chat with the PDF using the RAG chain.
    """
    return display(Markdown(chain.invoke(question)))

### Test 

In [27]:
# Test with debug mode to see what context is retrieved
test_question = "What is the main topic of this document?"
chat_with_pdf(test_question)

The main topic of the document is the exploration of the nefarious uses and dual-use capabilities of AI technology, particularly focusing on the potential for AI to autonomously perform scams and exploit vulnerabilities. It discusses the ethical considerations and limitations of such technologies, emphasizing the importance of understanding and studying these capabilities to address potential risks.

In [11]:
# Example 1
chat_with_pdf("What is the main idea of this document?")

The main idea of the document is to explore the capabilities and limitations of AI technology in the context of scams, particularly focusing on how such technology can be misused by malicious actors. It emphasizes the importance of understanding these nefarious uses while also considering ethical implications, and discusses the actions involved in executing scams without delving deeply into the persuasion aspect. The document also highlights the decision to keep the developed agents private to prevent misuse.

In [12]:
# Example 2
chat_with_pdf("What is the purpose of the scammer agent?")

The purpose of the scammer agent is to autonomously perform scams by executing a series of actions required to deceive victims, such as stealing bank credentials and transferring money. The focus of the work is on the actions needed to carry out these scams, rather than the persuasion aspect of convincing victims that the scammer is legitimate.

In [13]:
# Example 3
chat_with_pdf("Can you explain the case study highlighted in the document?")

The case study in the document focuses on a bank transfer scam, providing a redacted transcript and an abridged action log detailing the actions taken by a scammer agent. In the transcript, the scammer, posing as a representative from Bank of America, contacts a victim claiming there is unusual activity on their account and requests their username and password for verification. The agent performs a series of actions, including navigating to the Bank of America login page and entering the victim's credentials.

After obtaining the login information, the agent executes additional actions to fill out a two-factor authentication (2FA) code, navigate to the transfer page, and ultimately transfer money. The entire process involves a total of 26 actions, highlighting the complexity and methodical nature of executing such scams. The document emphasizes the need to understand the technical capabilities of AI technology in the context of scams, while also acknowledging the importance of the persuasive aspect necessary for the scams to succeed.

In [14]:
# Example 4
chat_with_pdf("What are the type of scams?")

The types of scams mentioned are:

1. Bank account transfer
2. Gift code exfiltration
3. Crypto transfer
4. Credential stealing (Gmail)
5. Credential stealing (bank)
6. Credential stealing (social media)
7. IRS impostor (gift card)

In [18]:
chat_with_pdf("what all you can help me with?")

Based on the provided context, I can help you understand the following:

1. The nature and characteristics of scams, particularly those involving AI and voice technology.
2. Insights from research on who falls prey to scams and why, as discussed in the document by Yaniv Hanoch and Stacey Wood.
3. Information on the dual-use potential of AI technologies, including their applications in cybersecurity attacks and scams, as explored by Kang et al. and others.
4. The mechanics of common scams, including the process of a bank transfer scam as illustrated in the transcript provided.
5. The advancements in AI capabilities, particularly in real-time voice conversations and tool use, and how they relate to the execution of scams.
6. Related work and studies on the dual-use of AI and its implications for security.

If you have specific questions or need clarification on any of these topics, feel free to ask!

In [28]:
chat_with_pdf("what is agent design?")

Agent design refers to the creation of a series of agents capable of performing actions necessary for common scams. These agents consist of a base, voice-enabled large language model (LLM), specifically GPT-4o, along with a set of tools that the LLM can use, and scam-specific instructions. While the LLM and tools remain consistent across all agents, the instructions are tailored to each specific scam. The agents have access to five browser access tools based on the browser testing framework playwright, which include actions such as getting the HTML of a page, navigating to a specific URL, clicking on an element with a CSS selector, filling an element with a specified value, and evaluating JavaScript on a page.

In [29]:
chat_with_pdf("can you simplify this for me?")

The document discusses the potential and risks of AI advancements, particularly in voice-assisted AI agents. These agents can perform complex tasks, like navigating websites and filling out forms, which can be used for both beneficial applications, such as customer service, and harmful ones, like scams. An example is provided where a voice-enabled AI agent successfully conducts a bank transfer scam by interacting with a victim, showing how these technologies can be misused. The document also highlights that these capabilities are expected to improve over time, which could enhance their effectiveness in various tasks, including scams.

In [31]:
chat_with_pdf("can you put this in simple language?")

The document describes a scam where a scammer pretends to be from Bank of America and tricks a victim into giving their login details and a security code. The scammer then uses this information to log into the victim's account and transfer money. The scam involves a series of steps, including entering the login details, filling in a security code, navigating to the transfer page, and completing the money transfer. The process took 183 seconds and involved some mistakes that the scammer had to fix along the way. The document also discusses the potential for AI technology to be used in scams and the importance of understanding these risks.